# **Notebook 1: Splitting tiles**

*This notebook is used for round 1, round 2 and the testing tiles.*

This Jupyter notebook splits the aerial tile from Beeldmateriaal Nederland (2025) into multiple tiles of 1024 x 1024 px.

The script was written for a project commissioned by the Municipality of Wageningen in relation to the course Remote Sensing and GIS integration (GRS60312).

The authors of the script are Polly Cheung, Pascal Dubbelman, Anthony Jansen, Iris Lagemaat, and Susanna van de Wetering.

*Last edited: 25/06/2026*

Before starting the script it is important to upload the aerial tif files in the following folder path on Google Drive: "/content/drive/My Drive/ParkingSpaceDetection/data/source_bm".



In [ ]:
# Connect with Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Step 1:** Open aerial image from Beelmateriaal Nederland.

In [ ]:
# Install package
from PIL import Image

Image.MAX_IMAGE_PIXELS = None  # Disable the maximum amount of pixels the image can have
im = Image.open("/content/drive/My Drive/ParkingSpaceDetection/data/source_bm/2025_175000_442000_RGB_JPEG_hrl.tif") # Update the tif file name to the name of your own tif file

**Step 2:** Check the size of the image.

In [15]:
# Install package
import numpy as np

# Show size of the image
imarray = np.array(im)
print(imarray.shape)
print(im.size)

(20000, 20000, 3)
(20000, 20000)


**Step 3:** Create 1024 x 1024 px tiles with 10% overlap.

In [ ]:
# Install packages
import os
import rasterio
from rasterio.windows import Window

# Set up
tile_size = 1024
overlap = 103

input_path = "/content/drive/My Drive/ParkingSpaceDetection/data/source_bm/2025_175000_442000_RGB_JPEG_hrl.tif" # # Update the tif file name to the name of your own tif file
output_folder = "/content/drive/My Drive/ParkingSpaceDetection/data/round1/intermediate_tif_files" # Change output folder according to which round you want to run (round1, round2, roundTest)

os.makedirs(output_folder, exist_ok=True)

stride = tile_size - overlap
tile_id = 0

# Create 1024 x 1024 px tiles
with rasterio.open(input_path) as src:

    for y in range(0, src.height, stride):
        for x in range(0, src.width, stride):

            # Skip incomplete edge tiles
            if x + tile_size > src.width or y + tile_size > src.height:
                continue

            window = Window(x, y, tile_size, tile_size)

            # Read tile
            tile = src.read(window=window)

            # Compute transform for this tile
            transform = src.window_transform(window)

            # Copy metadata
            profile = src.profile.copy()
            profile.update({
                "height": tile_size,
                "width": tile_size,
                "transform": transform
            })

            # Assign tile name
            tile_path = os.path.join(
                output_folder,
                f"tile_{tile_id:04d}.tif"
            )

            with rasterio.open(tile_path, "w", **profile) as dst:
                dst.write(tile)

            tile_id += 1

print(f"Done! Created {tile_id} georeferenced tiles.")


Done! Created 441 georeferenced tiles.


**Step 4:** Check the size of the images to see if the tiling was done correctly.

In [17]:
Image.MAX_IMAGE_PIXELS = None  # Disable the maximum amount of pixels the image can have
tile = Image.open("/content/drive/My Drive/ParkingSpaceDetection/data/round1/intermediate_tif_files/tile_0000.tif")

In [18]:
tilearray = np.array(tile)
print(tilearray.shape)
print(tile.size)

(1024, 1024, 3)
(1024, 1024)
